In [2]:
import bigram

step 0: train loss 4.7305, val loss 4.7241
step 300: train loss 2.8110, val loss 2.8249
step 600: train loss 2.5434, val loss 2.5682
step 900: train loss 2.4932, val loss 2.5088
step 1200: train loss 2.4863, val loss 2.5035
step 1500: train loss 2.4665, val loss 2.4921
step 1800: train loss 2.4683, val loss 2.4936
step 2100: train loss 2.4696, val loss 2.4846
step 2400: train loss 2.4638, val loss 2.4879
step 2700: train loss 2.4738, val loss 2.4911

od nos CAy go ghanoray t, co haringoudrou clethe k,LARof fr werar,
Is fa!


Thilemel cia h hmboomyorarifrcitheviPO, tle dst f qur'dig t cof boddo y t o ar pileas h mo wierl t,
S:
STENENEat I athe thounomy tinrent distesisanimald 3I: eliento ald, avaviconofrisist me Busarend un'soto vat s k,
SBRI he the f wendleindd t acoe ts ansu, thy ppr h.QULY:
KIIsqu pr odEd ch,
APrnes ouse bll owhored miner t ooon'stoume bupromo! fifoveghind hiarnge s.
MI aswimy or m, wardd tw'To tee abifewoetsphin sed The a


In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [2]:
torch.manual_seed(1337)
B,T,C = 4, 8, 2
x = torch.randn(B,T,C)
x.shape


torch.Size([4, 8, 2])

In [13]:
# x[b,t] = mean_(i<=t) x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] #(t, C)
        xbow[b,t] = torch.mean(xprev, 0)

In [ ]:
""" (B=1, T=3, C=2)

  x[0] = [[1.0, 2.0],     
          [3.0, 4.0],     
          [5.0, 6.0]]    

  逐个位置算 xbow:

  - t=0:xprev = x[0, :1] = [[1,2]](只有位置0)
  → mean = [1.0, 2.0]
  - t=1:xprev = x[0, :2] = [[1,2],[3,4]](位置0,1)
  → mean = [(1+3)/2, (2+4)/2] = [2.0, 3.0]
  - t=2:xprev = x[0, :3] = [[1,2],[3,4],[5,6]](位置0,1,2)
  → mean = [(1+3+5)/3, (2+4+6)/3] = [3.0, 4.0]

  结果:
  xbow[0] = [[1.0, 2.0],    # 只含自己
             [2.0, 3.0],    # 自己 + 前1个的平均
             [3.0, 4.0]]    # 自己 + 前2个的平均
"""

In [14]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [16]:
xbow[0] #each time step has info from past, although it's just a average

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [ ]:
torch.mean(x[0,:2], dim=0) #e.g. stay at b0, t2

tensor([-0.0894, -0.4926])

In [29]:
torch.tril(torch.ones((3,3)))

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

In [31]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3,3))
b = torch.randint(0, 10, (3,3)).float()
c = a @ b
print(a)
print(b)
print(c)


tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
tensor([[2., 7., 6.],
        [4., 6., 5.],
        [0., 4., 0.]])
tensor([[ 2.,  7.,  6.],
        [ 6., 13., 11.],
        [ 6., 17., 11.]])


In [16]:
torch.manual_seed(42) 
wei = torch.tril(torch.ones((T,T)))
xbow = torch.zeros((B,T,C))
for b in range(B):
    xbow[b] = wei @ x[b]

for b in range(B):
    for t in range(T):
        xbow[b,t] /= t + 1
xbow
    


tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])

In [32]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [38]:
sss = torch.ones(8,8)
sss = torch.tril(sss)
sss

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [41]:
print(sss);print(x[0])
print(sss @ x[0])

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])
tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])
tensor([[ 0.1808, -0.0700],
        [-0.1789, -0.9852],
        [ 0.4469, -0.9597],
        [ 1.4014, -0.8953],
        [ 1.7626,  0.2725],
        [ 0.4127, -0.2376],
        [ 0.6486, -0.4774],
        [-0.2725,  1.0659]])


In [42]:
xxx = sss @ x[0]

for i in range(8):
    xxx[i] /= i+1

xxx
    

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [33]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [46]:
#more elegant
#let tril normlized 
sss = torch.tril(torch.ones(8,8))
sss = sss / torch.sum(sss, dim=1, keepdim=True)
sss

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [48]:
print(sss @ x[0])


tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


In [51]:
# x[b,t] = mean_(i<=t) x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    sss = torch.tril(torch.ones((T,T)))
    sss /= torch.sum(sss, dim=1, keepdim=True)
    xbow[b] = sss @ x[b]
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [58]:
wei = torch.tril(torch.ones(T,T))
wei /= wei.sum(dim=1, keepdim=True)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [ ]:
xbow2 =  wei @ x #(T,T) @ (BTC) since broadcasting, wei-> (BTT)
xbow2 # BTC

tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])

In [87]:
#version 3
tril = torch.tril(torch.ones((T,T)))
wei = torch.zeros((T,T))
tril,wei

(tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
         [1., 1., 0., 0., 0., 0., 0., 0.],
         [1., 1., 1., 0., 0., 0., 0., 0.],
         [1., 1., 1., 1., 0., 0., 0., 0.],
         [1., 1., 1., 1., 1., 0., 0., 0.],
         [1., 1., 1., 1., 1., 1., 0., 0.],
         [1., 1., 1., 1., 1., 1., 1., 0.],
         [1., 1., 1., 1., 1., 1., 1., 1.]]),
 tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.]]))

In [33]:
shape = (T,T)
tril = torch.tril(torch.ones(shape))
wei = torch.zeros(shape)
wei.masked_fill_(tril == 0, float('-inf'))
F.softmax(wei, dim= -1)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [88]:
wei = wei.masked_fill(tril == 0, float('-inf')) 
wei

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

In [89]:
wei = F.softmax(wei, dim=-1)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [90]:
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

False

In [ ]:
#version 3
tril = torch.tril(torch.ones((T,T)))
wei = torch.zeros((T,T)) #interaction strength, how much of each token from the past we want to aggregate, 
# zero is set by us, which is a way to measure affinity, but cannot just be zero, it's data dependent, 

wei = wei.masked_fill(tril==0, float('-inf')) #only know info from new and past, not future 
wei = F.softmax(wei, dim=1) # exp and norm, ->1,0 and norm 
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [ ]:
"""
wei
[[1, 0, 0, 0],
   [1, 1, 0, 0],
   [1, 1, 1, 0],
   [1, 1, 1, 1]]
  - 作用:标记"每个位置能看到谁"。第 t 行的 1 = 位置 t 允许看的历史(含自己),0 =
  不能看的未来。
  
  wei = torch.zeros((T,T))
  - 交互强度矩阵,wei[t,i] = "位置 t 想从位置 i 聚合多少信息"。
  - 现在先全设为 0(下面重点解释为什么是 0)。

1. wei[t,i] 代表"亲和度 / affinity" —— 位置 t 和位置 i 有多"相关"、t 想从 i
  拿多少信息。

  2. 现在全设 0,softmax 后变成「等权平均」
  因为所有能看的位置 wei 都是 0 → exp(0)=1 都一样 →
  归一化后每个历史位置权重相等(1/n)。所以第 3 行 = [0.33, 0.33, 0.33, 0],就是"前 3
  个等权平均" —— 和上一版的下三角平均完全一样。

  3. 但"全 0"只是占位符,不是最终形态（你注释说的 "cannot just be zero, it's data
  dependent"）:
  - 全 0 意味着"每个历史 token 一样重要" —— 这太蠢了。真实情况里,不同 token
  的重要性不一样(比如预测下一个词时,某些前文关键、某些无关)。
  - 真正的 attention 里,wei 不是写死的 0,而是「根据数据算出来的」 —— 由每个 token
  的内容动态决定谁该关注谁。这就是 "data dependent"(依赖数据)。
  - 这一版先用 0 占位,跑通"掩码 + softmax + 加权"的框架;下一步把 0
  换成"算出来的亲和度",就变成完整的 self-attention。

  一句话:wei=0 =
  临时的「等权」占位,证明框架能工作;把它换成"数据算出的亲和度",框架立刻升级为真
  attention。
  
  So what is self-attention

  让每个 token 根据「内容」动态决定关注序列里的哪些其他token,并按关注度加权聚合它们的信息。
  
  假设序列是 5 个 token(为了直观,用词而不是字符):

  位置:   0      1      2      3       4
  token:  "the"  "cat"  "sat"  "on"   "it"

  现在模型处理到位置 4 "it",它要预测下一个词。"it" 指代谁?直觉上应该回头关注
  "cat"(it = 猫)。训练好的 attention 就会学到这一点。
  
  这个 0.55 就是关键 —— 训练让模型学会"当前是 it 时,应该重点回看
  cat"。聚合信息时(wei @ V),"it" 这个位置的新表示里混入了 55% 的 cat
  信息,于是它"知道"自己指的是猫。

  完整的学习后 wei(5×5,带因果掩码)

            the    cat    sat    on     it
  the   [ 1.00,   0,     0,     0,     0   ]   位置0 只能看自己
  cat   [ 0.30,  0.70,   0,     0,     0   ]   "cat" 主要看自己,略看 the
  sat   [ 0.10,  0.60,  0.30,   0,     0   ]   "sat" 关注施动者 cat
  on    [ 0.15,  0.25,  0.30,  0.30,   0   ]   介词,比较分散
  it    [ 0.05,  0.55,  0.10,  0.05,  0.25 ]   "it" 强烈关注 cat

  读这个矩阵的方法:
  - 每一行 = 一个 token 的"注意力分配"(和=1)。
  - 右上三角全 0 = 因果掩码,没人能看未来(你代码里 masked_fill(-inf) + softmax
  保证的)。
  - 权重高的地方 = 学到的"语义关联":cat←sat(施动)、cat←it(指代)。
  """

In [93]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [138]:

import torch
from torch import nn
from torch.nn import functional as F

batch_size = 32
block_size = 8
max_iter = 3000
eval_interval = 300
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iter = 200
n_embd = 32

torch.manual_seed(1337)

with open('../data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)

#map functions
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[ch] for ch in s] #take a string and map to list of ints
decode = lambda l: "".join([itos[i] for i in l]) #take a list of ints and map to string

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train = data[:n]
val = data[n:]

def get_batch(split):
    data = train if split == 'train' else val
    ix = torch.randint(len(data)-block_size, (batch_size,))
    xb = torch.stack([data[i:i+block_size] for i in ix])
    yb = torch.stack([data[i+1:i+block_size+1] for i in ix])
    xb, yb = xb.to(device), yb.to(device)
    return xb, yb  

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iter)
        for k in range(eval_iter):
            x, y = get_batch(split)
            logits, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out 

class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) #number of embedding dimensions
        self.position_embedding_table = nn.Embedding(block_size, n_embd) # each position has its own embd vec
        
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
    def forward(self, idx, targets=None):
        B, T = idx.shape
        # separate the embedding and output, now the embedding layer space is custom, 
        # which make attention, MLP calculation can calculate in a proper space, not a fixed vocab size
        
        # now, embedding -> x -> tok_emb, let tocken become a meaningful vector: understanding the input
        # lm_head -> tok_emb -> logits: translate the tok_emb to logits for predict the next token, :making output
        tok_emb = self.token_embedding_table(idx) #(BTC) C = n_embd
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) #(TC)
        x = tok_emb + pos_emb #now the token holds token identities and position it occur
        logits = self.lm_head(x) #BTC C = vocab_size
        
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        #idx is BT
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:] #BT = 8, since block size= 8 in the model, if don trunk the self(idx) cannot work
            logits, loss = self(idx_cond) #BTC
            logits = logits[:, -1, :] #BC
            probs = F.softmax(logits, dim=-1) #BC
            idx_next = torch.multinomial(probs, num_samples = 1) #B1
            idx = torch.cat([idx, idx_next], dim=1) #B T+1
        return idx

            

In [139]:
model = BigramLanguageModel()    
m = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

for iter in range(max_iter):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        
    xb, yb = get_batch('train')        
    
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.4801, val loss 4.4801
step 300: train loss 2.5404, val loss 2.5566
step 600: train loss 2.5160, val loss 2.5335
step 900: train loss 2.4967, val loss 2.5149
step 1200: train loss 2.5106, val loss 2.5254
step 1500: train loss 2.4853, val loss 2.5109
step 1800: train loss 2.4966, val loss 2.5198
step 2100: train loss 2.4949, val loss 2.5100
step 2400: train loss 2.4937, val loss 2.5102
step 2700: train loss 2.5040, val loss 2.5114


In [137]:
context = torch.zeros((1,1), dtype = torch.long, device = device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))
            
            


n he thoundeake hay FLLORikindoeravelye alvit an r,
haloven:
Why tod s:
t t,

W:
YCLe me

KI hirut t'thange tsph fagr s t y l, doriober mshese oupomorendomanell irdoloundoon,
Bulllixt?
I mo stare y's s t--f un muce thano whe, wha tes t longhy
Th rperpld t f ks, nithas.

THat d-lesinearersinetyorethed hin
Pre intindor arouto sod Handiny noo tromutandist onethish eve gst tllo sporur fo
Ho:
I th ster cisovomind migeueth suryovera no iren bucayousinour so'en,
FOMPen bus?

CAntonouco'sce w tir wissto


In [143]:
torch.manual_seed(42)
B,T,C = 4,8,32
x = torch.randn(B,T,C)

tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
out = wei @ x

out.shape

torch.Size([4, 8, 32])

In [ ]:
# Don't wannt the wei be uniform, we wannt each token konw which token might be more meaningful for them,
# so we can gather information from the past to find this affinity <- the problem self attention wannt to solve 

#Every single token at each position will emit two vectors
# query vector: what am i looking for 
# key vector: what do i contain 

#. the way we get the affinity between these tokens (keys @ querys) -> wei

In [ ]:
# Attention is a communication mechanism 
# self-attention <- the key, query, value comesfrom the same source x, it's x map three vectors, so 'self'

# Attention(Q, K, V) = softmax(Q@Kt / log(dk)) @ V
#log(dk) scaled attention

torch.manual_seed(42)
B,T,C = 4,8,32
x = torch.randn(B,T,C)

#single head perform self-attention
head_size = 16
key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)   #(B, T, 16)
q = query(x) #(B, T, 16)
wei = q @ k.transpose(-2,-1) #(B, T, 16) @ #(B, 16, T) -> (B,T,T)

tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
v = value(x)  #(B, T, 16)
out = wei @ v   # (B,T,T) @ #(B, T, 16)

out.shape

torch.Size([4, 8, 16])

In [161]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1905, 0.8095, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3742, 0.0568, 0.5690, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1288, 0.3380, 0.1376, 0.3956, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4311, 0.0841, 0.0582, 0.3049, 0.1217, 0.0000, 0.0000, 0.0000],
        [0.0537, 0.3205, 0.0694, 0.2404, 0.2568, 0.0592, 0.0000, 0.0000],
        [0.3396, 0.0149, 0.5165, 0.0180, 0.0658, 0.0080, 0.0373, 0.0000],
        [0.0165, 0.0375, 0.0144, 0.1120, 0.0332, 0.4069, 0.3136, 0.0660]],
       grad_fn=<SelectBackward0>)

In [155]:
(q @ k.transpose(1, 2)).shape

torch.Size([4, 8, 8])

In [ ]:
#log(dk) scaled attention -> norm 
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei =  q @ k.transpose(-2, -1)
k.var(), q.var(), wei.var()

(tensor(1.0392), tensor(0.9791), tensor(12.4662))

In [170]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei =  q @ k.transpose(-2, -1) * head_size**-0.5
k.var(), q.var(), wei.var()

(tensor(1.0346), tensor(0.9618), tensor(0.9446))

In [ ]:
#temperature
# The reason is to scale is softmax
#if the input is sharp, the softmax will make like one-hot weight
# we don't wannt softmax to focu on sigle node in initial part, diffusion is our goal 
torch.softmax(torch.tensor([0.1,0.2,0.3,0.4,0.5]), dim=-1)

tensor([0.1621, 0.1792, 0.1980, 0.2188, 0.2419])

In [178]:
torch.softmax(torch.tensor([0.1,0.2,0.3,0.4,0.5])*15, dim=-1)


tensor([0.0019, 0.0086, 0.0387, 0.1734, 0.7773])

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) #TT
        
    def forward(self, x):
        k = self.key(x) #BTC
        q = self.key(x) #BTC
        v = self.key(x) #BTC
        wei = k @ q.transpose(-2, -1) * C**-0.5  #BTT
        wei = wei.masked_fill(self.tril[:T,:T], float('-inf'))
        wei = F.softmax(wei, dim= -1)
        out = wei @ v
        return v
        
        
        
    

In [ ]:
class Head(nn.Module):
    "one head of self-attention"
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) #tril is not a parameter, put in buffer
    
    def forward(self, x):
        B, T, C = x.shape 
        k = self.key(x)     #BT C = head_size
        q = self.query(x)   #BT C = head_size
        
        #compute attention "affinities"
        wei = q @ k.transpose(-2, -1)  * C**-0.5  #BTT
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
                
        v = self.value(x)   #BT C = head_size
        out = wei @ v #BTT @  #BT C ->  #BT C = head_size
        
        return out 
        
        
        
        

In [189]:
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) #number of embedding dimensions
        self.position_embedding_table = nn.Embedding(block_size, n_embd) # each position has its own embd vec
        self.sa_head = Head(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx) #(BTC) C = n_embd
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) #(TC)
        x = tok_emb + pos_emb #now the token holds token identities and position it occur
                
        x = self.sa_head(x) #The thinking part from the previous tocken, not just self 
        
        logits = self.lm_head(x) #BTC C = vocab_size
        
        
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        #idx is BT
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:] #BT = 8, since block size= 8 in the model, if don trunk the self(idx) cannot work
            logits, loss = self(idx_cond) #BTC
            logits = logits[:, -1, :] #BC
            probs = F.softmax(logits, dim=-1) #BC
            idx_next = torch.multinomial(probs, num_samples = 1) #B1
            idx = torch.cat([idx, idx_next], dim=1) #B T+1
        return idx

            

In [190]:
batch_size = 32
block_size = 8
max_iter = 5000
eval_interval = 300
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iter = 200
n_embd = 32

In [191]:
model = BigramLanguageModel()    
m = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

for iter in range(max_iter):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        
    xb, yb = get_batch('train')        
    
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.2409, val loss 4.2410
step 300: train loss 2.8117, val loss 2.8452
step 600: train loss 2.6131, val loss 2.6384
step 900: train loss 2.5223, val loss 2.5415
step 1200: train loss 2.4890, val loss 2.4937
step 1500: train loss 2.4615, val loss 2.4729
step 1800: train loss 2.4621, val loss 2.4611
step 2100: train loss 2.4267, val loss 2.4599
step 2400: train loss 2.4180, val loss 2.4476
step 2700: train loss 2.4207, val loss 2.4369
step 3000: train loss 2.4177, val loss 2.4170
step 3300: train loss 2.3980, val loss 2.4186
step 3600: train loss 2.4039, val loss 2.4207
step 3900: train loss 2.3892, val loss 2.4043
step 4200: train loss 2.4013, val loss 2.4054
step 4500: train loss 2.3900, val loss 2.4212
step 4800: train loss 2.3888, val loss 2.4003


In [204]:
context = torch.zeros((1,1), dtype = torch.long, device = device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))
            


WArwises airt ucry isen atharithus fisten? thereghticer yu f dyu utzerd
Yo Bun en, ly, ame mus bimespyour buds hanowrolld henof nquube hal thavenwas dary 'lorme,
O:
S:
Ar ay verd irtewanovewe; ayo ithesere

Whendere non theak!
'Til;
And a sheapa me ofronouriitro,
D ont hing
Sesout thed he yak wem too

Gofel fo red ove.
hin touseveit eso str felld wito, turg mangir, lit thelste Matrme howy if th rd wal
Tet sithed yove quuler abry Ben dacis.
MAnele ldaby brw-pee fouge: tly touleld I on taved, wa u
